## Notebook 06 — LSTM Model

### Objective

Train an LSTM neural network on sequences of 20 days
of features per ticker. Generate OOF predictions
for stacking in notebook 07.

### Why LSTM Adds Value to the Ensemble

XGBoost and LightGBM see one snapshot at a time.
LSTM sees the entire 20-day journey leading up to today.

Patterns LSTM can find that tree models cannot:
  RSI rising for 5 consecutive days
  Volume building gradually over 2 weeks
  MACD trending in one direction for a month
  Price respecting a moving average across 10 sessions

The ensemble combines:
  Tree models  : strong on point-in-time signals
  LSTM         : strong on sequence patterns
  Meta-model   : learns when to trust which model

### Notebook Structure

  Cell 1  →  Imports and configuration
  Cell 2  →  Load and verify data
  Cell 3  →  Train / Validation split
  Cell 4  →  Feature scaling
  Cell 5  →  Sequence building function
  Cell 6  →  Build train and validation sequences
  Cell 7  →  Sequence sanity validation
  Cell 8  →  LSTM architecture
  Cell 9  →  Walk-forward OOF loop
  Cell 10 →  OOF sanity validation
  Cell 11 →  Train final LSTM
  Cell 12 →  Evaluate on validation
  Cell 13 →  Save model scaler and OOF predictions

### Key Constants

  LOOKBACK = 20    each sequence is 20 days of history
                   matches MA20 window from features
                   one trading month

  BATCH_SIZE = 256 rows per training batch
                   smaller = slower but more stable
                   larger = faster but less robust
                   256 is a good balance

  MAX_EPOCHS = 100 maximum training passes
                   early stopping cuts it short
                   if validation loss stops improving

  PATIENCE = 10    epochs without improvement before stopping
                   prevents overfitting automatically

In [1]:
# ============================================================
# CELL 1 - Imports and Configuration
# ============================================================
# Import all libraries needed for LSTM training
# Fix all random seeds — TensorFlow, NumPy, Python
# Configure TensorFlow to use available GPU if any
# Suppress TensorFlow verbose logging
# ============================================================

import os
# Suppress TensorFlow warnings before import
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pandas as pd
import numpy as np
import tensorflow as tf
from   tensorflow import keras
from   tensorflow.keras import layers
from   tensorflow.keras.callbacks import EarlyStopping
from   sklearn.preprocessing      import MinMaxScaler
from   sklearn.model_selection    import TimeSeriesSplit
from   sklearn.metrics            import (classification_report,
                                          balanced_accuracy_score,
                                          confusion_matrix,
                                          f1_score)
from   sklearn.utils.class_weight import compute_sample_weight
import joblib
import random
import warnings
warnings.filterwarnings('ignore')

# ── Fix all random seeds for reproducibility ──────────────
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Configure deterministic operations where possible
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)

# ── Project constants — must match notebooks 04 and 05 ────
PURGE_DAYS = 5    # must match prediction horizon
N_SPLITS   = 5    # walk-forward CV folds
N_CLASSES  = 3    # Sell=0, Hold=1, Buy=2

# ── LSTM-specific constants ───────────────────────────────
LOOKBACK    = 20    # 20 days of history per sequence
BATCH_SIZE  = 256   # training batch size
MAX_EPOCHS  = 100   # max epochs, early stopping cuts short
PATIENCE    = 10    # early stopping patience

# Class label map
LABELS = {0: 'Sell', 1: 'Hold', 2: 'Buy'}

# File paths
PROCESSED_PATH = '../data/processed/all_stocks_features.csv'
MODELS_DIR     = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

# ── Verify TensorFlow installation ────────────────────────
print('Environment ready')
print('=' * 50)
print(f'  TensorFlow version : {tf.__version__}')
print(f'  Keras version      : {keras.__version__}')

# Check for GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'  GPU available      : {len(gpus)} device(s)')
    print(f'  Will use           : GPU acceleration')
    for gpu in gpus:
        print(f'    {gpu.name}')
else:
    print(f'  GPU available      : No')
    print(f'  Will use           : CPU (slower but works)')

print()
print('Configuration:')
print(f'  Random state       : {RANDOM_STATE}')
print(f'  Purge days         : {PURGE_DAYS}')
print(f'  CV folds           : {N_SPLITS}')
print(f'  Classes            : {LABELS}')
print()
print('LSTM-specific:')
print(f'  Lookback window    : {LOOKBACK} days')
print(f'  Batch size         : {BATCH_SIZE}')
print(f'  Max epochs         : {MAX_EPOCHS}')
print(f'  Early stop patience: {PATIENCE}')
print()
print(f'  Models directory   : {MODELS_DIR}')

Environment ready
  TensorFlow version : 2.21.0
  Keras version      : 3.14.1
  GPU available      : No
  Will use           : CPU (slower but works)

Configuration:
  Random state       : 42
  Purge days         : 5
  CV folds           : 5
  Classes            : {0: 'Sell', 1: 'Hold', 2: 'Buy'}

LSTM-specific:
  Lookback window    : 20 days
  Batch size         : 256
  Max epochs         : 100
  Early stop patience: 10

  Models directory   : ../models/


-----------

### Cell 2 — Load and Verify Processed Data

#### Identical to Notebooks 04 and 05

Same processed file. Same drops. Same assertions.
Same feature_cols.pkl loaded from disk.

This is critical for ensemble consistency:
  All three models train on the exact same dataset
  All three models use the exact same 17 features
  All three models split at the exact same date boundary

Any deviation breaks the stacking layer in notebook 07.

In [2]:
# ============================================================
# CELL 2 - Load and Verify Processed Data
# ============================================================
# Identical load logic to notebooks 04 and 05
# Drop macd and macd_signal — same as before
# Load FEATURE_COLS from disk — guarantees consistency
# ============================================================

# Load processed data
df = pd.read_csv(PROCESSED_PATH, parse_dates=['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print('Processed data loaded')
print('=' * 50)
print(f'  Shape      : {df.shape}')
print(f'  NaNs       : {df.isnull().sum().sum()}')
print(f'  Tickers    : {df["Ticker"].nunique()}')
print(f'  Date range : {df["Date"].min().date()} '
      f'to {df["Date"].max().date()}')

# Hard assertions
assert df.shape[0]             == 29100
assert df.shape[1]             == 27
assert df.isnull().sum().sum() == 0
assert df['Ticker'].nunique()  == 20

print('\nAll load assertions passed ✅')

# Drop redundant MACD columns
df = df.drop(columns=['macd', 'macd_signal'])

assert df.shape[1]     == 25
assert 'macd'          not in df.columns
assert 'macd_signal'   not in df.columns
assert 'macd_hist'     in df.columns

print(f'\nDropped : macd, macd_signal')
print(f'Shape after drop : {df.shape}')
print('Column drop verified ✅')

# Load feature columns from disk
FEATURE_COLS = joblib.load(f'{MODELS_DIR}feature_cols.pkl')
TARGET_COL   = 'target'

print(f'\nFeature columns loaded from disk')
print(f'  Source   : {MODELS_DIR}feature_cols.pkl')
print(f'  Features : {len(FEATURE_COLS)}')

# Verify all features exist
missing = [c for c in FEATURE_COLS if c not in df.columns]
assert len(missing) == 0, f'Missing: {missing}'
assert len(FEATURE_COLS) == 17

print('\nAll feature assertions passed ✅')

Processed data loaded
  Shape      : (29100, 27)
  NaNs       : 0
  Tickers    : 20
  Date range : 2019-03-14 to 2024-12-20

All load assertions passed ✅

Dropped : macd, macd_signal
Shape after drop : (29100, 25)
Column drop verified ✅

Feature columns loaded from disk
  Source   : ../models/feature_cols.pkl
  Features : 17

All feature assertions passed ✅


------

### Cell 3 — Train / Validation Split by Date

#### Identical Boundary to Notebooks 04 and 05

  Train      :  2019-03-14  to  2023-12-31
  Validation :  2024-01-01  to  2024-12-20

Same split. Same reset_index. Same assertions.
Same row counts — 24,180 train and 4,920 validation.

#### One LSTM-Specific Note

LSTM needs 20 days of history to form its first sequence.
This means the first 19 rows per ticker in the training
set cannot be the TARGET row of any sequence.
They can only appear as CONTEXT rows in sequences.

This is handled in Cell 5 sequence building function.
We note it here so the row count reduction in Cell 6
does not come as a surprise.

Expected sequence counts after building:
  Train sequences  :  24,180 - (19 × 20 tickers)
                   =  24,180 - 380
                   =  ~23,800 sequences

  Val sequences    :  4,920 - (19 × 20 tickers)
                   =  4,920 - 380
                   =  ~4,540 sequences

The 19 rows lost per ticker are the warmup rows
needed to form the first complete 20-day window.

In [3]:
# ============================================================
# CELL 3 - Train / Validation Split by Date
# ============================================================
# Identical split to notebooks 04 and 05
# Same boundary, same reset_index, same assertions
# Same row counts verified with hard assertions
# ============================================================

TRAIN_END = '2023-12-31'
VAL_START = '2024-01-01'

df_train = df[df['Date'] <= TRAIN_END].copy()
df_val   = df[df['Date'] >= VAL_START].copy()

# Reset index — critical for sequence building alignment
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)

print('Train / Validation split complete')
print('=' * 55)
print(f'  Train      : {df_train["Date"].min().date()} '
      f'to {df_train["Date"].max().date()} '
      f'| {len(df_train):,} rows '
      f'({len(df_train)/len(df)*100:.1f}%)')
print(f'  Validation : {df_val["Date"].min().date()} '
      f'to {df_val["Date"].max().date()} '
      f'| {len(df_val):,} rows '
      f'({len(df_val)/len(df)*100:.1f}%)')
print(f'  Total      : {len(df_train) + len(df_val):,} rows')
print()
print('  Live 2025  : downloaded from yfinance in dashboard')
print('  Live 2026  : real-time predictions in production')

# Core assertions
assert df_train['Date'].max() < df_val['Date'].min(), \
    'DATE OVERLAP — LEAKAGE'
assert df_train['Ticker'].nunique() == 20
assert df_val['Ticker'].nunique()   == 20
assert len(df_train) + len(df_val)  == len(df)
assert df_train.index[0]            == 0
assert df_train.index[-1]           == len(df_train) - 1

# Verify matches notebooks 04 and 05 exactly
assert len(df_train) == 24180, \
    f'Train size {len(df_train)} differs from nb04/05'
assert len(df_val)   == 4920, \
    f'Val size {len(df_val)} differs from nb04/05'

print()
print('Split assertions passed ✅')
print('Split matches notebooks 04 and 05 exactly ✅')
print()

# LSTM sequence count preview
train_seqs = len(df_train) - (LOOKBACK - 1) * 20
val_seqs   = len(df_val)   - (LOOKBACK - 1) * 20
print('LSTM Sequence Count Preview:')
print(f'  Lookback window   : {LOOKBACK} days')
print(f'  Warmup rows lost  : {(LOOKBACK-1)*20} '
      f'({LOOKBACK-1} rows × 20 tickers)')
print(f'  Train sequences   : ~{train_seqs:,}')
print(f'  Val sequences     : ~{val_seqs:,}')
print()

# Class distribution
print('Class distribution comparison:')
print(f'  {"Class":<8} {"Train %":>8} {"Val %":>8} '
      f'{"Difference":>12}')
print(f'  {"-" * 40}')
for cls in range(N_CLASSES):
    train_pct = (df_train['target'] == cls).mean() * 100
    val_pct   = (df_val['target']   == cls).mean() * 100
    diff      = val_pct - train_pct
    label     = LABELS[cls]
    flag      = '⚠️' if abs(diff) > 5 else '✅'
    print(f'  {label:<8} {train_pct:>8.1f}% '
          f'{val_pct:>8.1f}% {diff:>+11.1f}%  {flag}')

print()
print('Validation set is now LOCKED until Cell 12 ✅')

Train / Validation split complete
  Train      : 2019-03-14 to 2023-12-29 | 24,180 rows (83.1%)
  Validation : 2024-01-02 to 2024-12-20 | 4,920 rows (16.9%)
  Total      : 29,100 rows

  Live 2025  : downloaded from yfinance in dashboard
  Live 2026  : real-time predictions in production

Split assertions passed ✅
Split matches notebooks 04 and 05 exactly ✅

LSTM Sequence Count Preview:
  Lookback window   : 20 days
  Warmup rows lost  : 380 (19 rows × 20 tickers)
  Train sequences   : ~23,800
  Val sequences     : ~4,540

Class distribution comparison:
  Class     Train %    Val %   Difference
  ----------------------------------------
  Sell         24.9%     20.7%        -4.2%  ✅
  Hold         42.5%     45.9%        +3.5%  ✅
  Buy          32.6%     33.4%        +0.8%  ✅

Validation set is now LOCKED until Cell 12 ✅


-----------

### Cell 4 — Feature Scaling with MinMaxScaler

#### Why Tree Models Did Not Need Scaling

XGBoost and LightGBM make decisions by splitting
on thresholds. They ask:
  "Is RSI > 70?"
  "Is volume_ratio > 2.0?"

The actual numeric scale does not matter.
Doubling all values changes nothing because
the threshold also doubles proportionally.

#### Why LSTM Absolutely Needs Scaling

LSTM uses gradient descent to learn.
It multiplies features by weights and adds them.
Features on different scales create a problem:

  rsi_14        : range  9   to  94    (scale ~85)
  return_1d     : range -0.35 to +0.26 (scale ~0.6)
  volume_ratio  : range  0.14 to 11.4  (scale ~11)

During backpropagation gradients flow backwards.
Large-scale features (rsi_14) produce large gradients.
Small-scale features (return_1d) produce tiny gradients.

The model adjusts weights proportional to gradient size.
Result: LSTM learns almost entirely from rsi_14
        and nearly ignores return_1d.
        This wastes 16 out of 17 features.

After scaling everything to range 0 to 1:

  rsi_14       : 0.0 to 1.0

  return_1d    : 0.0 to 1.0
  
  volume_ratio : 0.0 to 1.0

All features contribute equally to learning.
The LSTM can find patterns across all 17 signals.

#### Why We Fit on Training Data Only

MinMaxScaler computes:

  min value = minimum of the column

  max value = maximum of the column

  scaled    = (value - min) / (max - min)

If we fit on the full dataset:
  The scaler sees 2024 validation data
  It learns the min and max of 2024
  Training transformation uses future knowledge
  This is data leakage — forbidden

Correct approach:
  Fit   on df_train only  → scaler learns 2019-2023 range
  Transform df_train      → scale training data
  Transform df_val        → scale with TRAINING min/max

If 2024 has a value outside the 2019-2023 range
the scaler will produce a value below 0 or above 1.
This is correct and expected — it means 2024 had
a more extreme value than anything in training.
The model handles this gracefully.

#### What We Scale

We scale ALL 17 features at the row level
before building sequences.

The target column is NOT scaled.
It contains class labels 0, 1, 2 — not features.

#### Saving the Scaler

The scaler is saved to disk alongside the model.
The dashboard must use the SAME fitted scaler
when processing live 2025 and 2026 data.

If the dashboard fits a new scaler on live data:
  Different min/max → different scaled values
  Model receives inputs it was never trained on
  Predictions are wrong silently

In [4]:
# ============================================================
# CELL 4 - Feature Scaling with MinMaxScaler
# ============================================================
# Fit scaler on df_train ONLY — never on validation
# Transform both train and val with the SAME fitted scaler
# Save scaler to disk — dashboard uses same scaler
# Verify scaling is correct — all values in range 0 to 1
# for training data, may go outside for validation
# ============================================================

# ── Fit scaler on training data only ──────────────────────
scaler = MinMaxScaler()

# Extract feature matrix from training set
X_train_raw = df_train[FEATURE_COLS].values
X_val_raw   = df_val[FEATURE_COLS].values

# Fit ONLY on training — learns min/max from 2019-2023
scaler.fit(X_train_raw)

# Transform both sets with the SAME fitted scaler
X_train_scaled = scaler.transform(X_train_raw)
X_val_scaled   = scaler.transform(X_val_raw)

print('Feature scaling complete')
print('=' * 55)
print(f'  Scaler type       : MinMaxScaler')
print(f'  Fitted on         : training set only (2019-2023)')
print(f'  Features scaled   : {len(FEATURE_COLS)}')
print(f'  Train shape       : {X_train_scaled.shape}')
print(f'  Val shape         : {X_val_scaled.shape}')
print()

# ── Verify training scaling ────────────────────────────────
train_min = X_train_scaled.min()
train_max = X_train_scaled.max()
print('Training set scaling verification:')
print(f'  Min value : {train_min:.6f}  (must be exactly 0.0)')
print(f'  Max value : {train_max:.6f}  (must be exactly 1.0)')
assert abs(train_min) < 1e-10, \
    f'Train min {train_min} != 0.0 — scaling failed'
assert abs(train_max - 1.0) < 1e-10, \
    f'Train max {train_max} != 1.0 — scaling failed'
print('  Training scaling verified ✅')
print()

# ── Check validation scaling ──────────────────────────────
val_min = X_val_scaled.min()
val_max = X_val_scaled.max()
print('Validation set scaling check:')
print(f'  Min value : {val_min:.6f}')
print(f'  Max value : {val_max:.6f}')
if val_min < 0 or val_max > 1:
    print('  ℹ️  Values outside 0-1 range detected')
    print('     This means 2024 had more extreme values')
    print('     than 2019-2023 training period')
    print('     This is expected and handled correctly')
    n_below = (X_val_scaled < 0).sum()
    n_above = (X_val_scaled > 1).sum()
    print(f'     Values below 0 : {n_below}')
    print(f'     Values above 1 : {n_above}')
else:
    print('  ✅ All validation values within 0-1 range')
print()

# ── Per feature scaling summary ───────────────────────────
print('Per Feature Scaling Summary:')
print(f'  {"Feature":<25} {"Train Min":>10} '
      f'{"Train Max":>10} {"Val Min":>10} {"Val Max":>10}')
print(f'  {"-" * 70}')
for i, feat in enumerate(FEATURE_COLS):
    t_min = X_train_scaled[:, i].min()
    t_max = X_train_scaled[:, i].max()
    v_min = X_val_scaled[:, i].min()
    v_max = X_val_scaled[:, i].max()
    flag  = ''
    if v_min < -0.1 or v_max > 1.1:
        flag = ' ⚠️'
    print(f'  {feat:<25} {t_min:>10.4f} '
          f'{t_max:>10.4f} {v_min:>10.4f} {v_max:>10.4f}{flag}')

print()

# ── Save scaler to disk ───────────────────────────────────
PATH_SCALER = f'{MODELS_DIR}lstm_scaler.pkl'
joblib.dump(scaler, PATH_SCALER)
size_scaler = os.path.getsize(PATH_SCALER) / 1024
print(f'Scaler saved to : {PATH_SCALER}')
print(f'Scaler size     : {size_scaler:.1f} KB')

# Verify scaler reloads correctly
scaler_loaded  = joblib.load(PATH_SCALER)
X_test_orig    = scaler.transform(X_val_raw[:5])
X_test_loaded  = scaler_loaded.transform(X_val_raw[:5])
assert np.allclose(X_test_orig, X_test_loaded), \
    'SCALER VERIFY FAILED: reloaded scaler differs'
print('Scaler verified : reload produces same output ✅')
print()
print('Note: dashboard must load this scaler file')
print('      Never fit a new scaler on live data')

Feature scaling complete
  Scaler type       : MinMaxScaler
  Fitted on         : training set only (2019-2023)
  Features scaled   : 17
  Train shape       : (24180, 17)
  Val shape         : (4920, 17)

Training set scaling verification:
  Min value : 0.000000  (must be exactly 0.0)
  Max value : 1.000000  (must be exactly 1.0)
  Training scaling verified ✅

Validation set scaling check:
  Min value : 0.000000
  Max value : 1.000000
  ✅ All validation values within 0-1 range

Per Feature Scaling Summary:
  Feature                    Train Min  Train Max    Val Min    Val Max
  ----------------------------------------------------------------------
  return_1d                     0.0000     1.0000     0.2515     0.9325
  return_5d                     0.0000     1.0000     0.1999     0.8760
  return_10d                    0.0000     1.0000     0.2347     0.8113
  return_20d                    0.0000     1.0000     0.1840     0.7231
  price_vs_ma20                 0.0000     1.0000     0

----------------------

### Cell 5 — Sequence Building Function

#### What a Sequence Is

For the LSTM each sample is not one row.
It is a window of 20 consecutive rows.

Single row (tree model input):
  [rsi=65, macd=0.3, volume_ratio=1.8, ...]
  Shape: (17,)

Sequence (LSTM input):
  Day 1  : [rsi=52, macd=-0.1, volume_ratio=0.9, ...]

  Day 2  : [rsi=55, macd=-0.0, volume_ratio=1.1, ...]

  ...
  Day 20 : [rsi=65, macd=0.3,  volume_ratio=1.8, ...]

  Shape: (20, 17)

The LSTM reads all 20 days and predicts what
happens on day 21.

#### Why Per Ticker Groupby Is Mandatory

This is Rule 3 from the project implementation guide.

The dataframe has 20 tickers stacked vertically.

If we build sequences naively across all rows:

  Row 1508  AAPL  2023-12-27

  Row 1509  AAPL  2023-12-29   ← AAPL ends here

  Row 1510  ADBE  2019-03-14   ← ADBE starts here

  Naive sequence at position 1510:

    Days 1-19 : AAPL Dec 2023 data  ← wrong ticker

    Day 20    : ADBE Mar 2019 data  ← correct ticker

    Target    : ADBE Mar 2019 label ← predicting ADBE
                                      using AAPL history



  This is silent corruption.
  No error is raised.
  The LSTM trains on nonsense.

  Correct approach using groupby:

    Build sequences separately for each ticker

    First 19 rows of each ticker cannot be target rows

    They can only appear as context in other sequences

#### Function Design

The build_sequences function takes:

  data     : scaled feature array (N rows, 17 features)

  targets  : target labels array (N rows,)

  dates    : date array (N rows,)

  tickers  : ticker array (N rows,)
  
  lookback : window size (20)

It returns:
  X    : sequence array (samples, 20, 17)
  y    : target array (samples,)
  meta : dataframe with Date and Ticker per sequence
         used for OOF alignment and debugging

The meta dataframe is critical for OOF alignment.
It tells us exactly which Date and Ticker each
sequence corresponds to so we can store OOF
predictions in the right position.

#### Sequence Count Verification

For each ticker with N rows:
  Number of sequences = N - lookback + 1
  First 19 rows are context only
  Row 20 onwards is a valid target row

Total sequences = sum across all 20 tickers
               = 20 × (rows_per_ticker - 19)

In [5]:
# ============================================================
# CELL 5 - Sequence Building Function
# ============================================================
# Build 3D sequence arrays for LSTM input
# Must be called inside groupby(Ticker) loop
# Never build sequences across ticker boundaries
# Returns X (samples,20,17), y (samples,), meta (DataFrame)
# meta tracks Date and Ticker for OOF alignment
# ============================================================

def build_sequences(scaled_features, targets,
                    dates, tickers, lookback=20):
    """
    Build LSTM sequences per ticker.

    Parameters:
        scaled_features : np.array (N, 17) scaled features
        targets         : np.array (N,)    target labels
        dates           : np.array (N,)    dates
        tickers         : np.array (N,)    ticker symbols
        lookback        : int              sequence length

    Returns:
        X    : np.array (samples, lookback, n_features)
        y    : np.array (samples,)
        meta : pd.DataFrame with Date and Ticker columns
    """
    X_list    = []
    y_list    = []
    meta_list = []

    unique_tickers = np.unique(tickers)

    for ticker in unique_tickers:
        # Get mask for this ticker
        mask = tickers == ticker

        # Extract this ticker's data in chronological order
        ticker_features = scaled_features[mask]
        ticker_targets  = targets[mask]
        ticker_dates    = dates[mask]
        n_rows          = len(ticker_features)

        # Build sequences for this ticker only
        # i starts at lookback so sequence i-lookback:i
        # is always entirely within this ticker's data
        for i in range(lookback, n_rows + 1):
            # Sequence: 20 days of features
            X_list.append(ticker_features[i-lookback:i])
            # Target: label for the LAST day in sequence
            y_list.append(ticker_targets[i-1])
            # Meta: date and ticker for this sequence
            meta_list.append({
                'Date'  : ticker_dates[i-1],
                'Ticker': ticker
            })

    X    = np.array(X_list,  dtype=np.float32)
    y    = np.array(y_list,  dtype=np.int32)
    meta = pd.DataFrame(meta_list)

    return X, y, meta


# ── Verify function with a small test ────────────────────
print('Sequence Building Function Defined')
print('=' * 55)
print('Running verification test on AAPL and ADBE only...')
print()

# Test with first two tickers only
test_mask     = df_train['Ticker'].isin(['AAPL', 'ADBE'])
test_features = X_train_scaled[test_mask]
test_targets  = df_train.loc[test_mask,
                             'target'].values.astype(int)
test_dates    = df_train.loc[test_mask, 'Date'].values
test_tickers  = df_train.loc[test_mask, 'Ticker'].values

X_test, y_test, meta_test = build_sequences(
    test_features, test_targets,
    test_dates,    test_tickers,
    lookback=LOOKBACK
)

print(f'  Test tickers      : AAPL and ADBE only')
print(f'  Input rows        : {test_mask.sum():,}')
print(f'  Output X shape    : {X_test.shape}')
print(f'  Output y shape    : {y_test.shape}')
print(f'  Output meta shape : {meta_test.shape}')
print()

# ── Ticker boundary verification ──────────────────────────
# Verify sequences are built per ticker not across tickers
# Every entry in meta must belong to exactly one ticker
# No sequence should mix rows from different tickers
aapl_meta = meta_test[meta_test['Ticker'] == 'AAPL']
adbe_meta = meta_test[meta_test['Ticker'] == 'ADBE']

assert len(aapl_meta) > 0, \
    'No AAPL sequences found'
assert len(adbe_meta) > 0, \
    'No ADBE sequences found'
assert meta_test['Ticker'].nunique() == 2, \
    'More than 2 tickers in meta — boundary crossed'
assert len(aapl_meta) + len(adbe_meta) == len(meta_test), \
    'Sequence count mismatch — ticker assignment wrong'

print('Ticker boundary verification:')
print(f'  AAPL sequences          : {len(aapl_meta)}')
print(f'  ADBE sequences          : {len(adbe_meta)}')
print(f'  Total sequences         : {len(meta_test)}')
print(f'  Unique tickers in meta  : '
      f'{meta_test["Ticker"].nunique()}  (must be 2)')
print(f'  Ticker boundary check   : no crossings ✅')
print()

# ── Shape verification ────────────────────────────────────
rows_per_ticker = test_mask.sum() // 2
expected_seqs   = 2 * (rows_per_ticker - LOOKBACK + 1)
assert X_test.shape[0] == expected_seqs, \
    f'Expected {expected_seqs} sequences got {X_test.shape[0]}'
assert X_test.shape[1] == LOOKBACK, \
    f'Expected {LOOKBACK} timesteps got {X_test.shape[1]}'
assert X_test.shape[2] == len(FEATURE_COLS), \
    f'Expected {len(FEATURE_COLS)} features ' \
    f'got {X_test.shape[2]}'

print('Shape verification:')
print(f'  X shape : {X_test.shape}')
print(f'           (sequences, timesteps, features)  ✅')
print(f'  y shape : {y_test.shape}  ✅')
print()

# ── Value range verification ──────────────────────────────
print('Value range verification:')
print(f'  X min           : {X_test.min():.4f}  '
      f'(training data should be ~0.0)')
print(f'  X max           : {X_test.max():.4f}  '
      f'(training data should be ~1.0)')
print(f'  y unique values : {np.unique(y_test)}  '
      f'(must be 0, 1, 2)')

assert set(np.unique(y_test)) <= {0, 1, 2}, \
    f'Unexpected target values: {np.unique(y_test)}'
print()

print('=' * 55)
print('Sequence building function verified ✅')
print('Ready to build full sequences in Cell 6')
print()
print('Function signature:')
print('  build_sequences(scaled_features, targets,')
print('                  dates, tickers, lookback=20)')
print('  Returns: X    shape (samples, 20, 17)')
print('           y    shape (samples,)')
print('           meta shape (samples, 2) — Date, Ticker')

Sequence Building Function Defined
Running verification test on AAPL and ADBE only...

  Test tickers      : AAPL and ADBE only
  Input rows        : 2,418
  Output X shape    : (2380, 20, 17)
  Output y shape    : (2380,)
  Output meta shape : (2380, 2)

Ticker boundary verification:
  AAPL sequences          : 1190
  ADBE sequences          : 1190
  Total sequences         : 2380
  Unique tickers in meta  : 2  (must be 2)
  Ticker boundary check   : no crossings ✅

Shape verification:
  X shape : (2380, 20, 17)
           (sequences, timesteps, features)  ✅
  y shape : (2380,)  ✅

Value range verification:
  X min           : 0.0000  (training data should be ~0.0)
  X max           : 1.0000  (training data should be ~1.0)
  y unique values : [0 1 2]  (must be 0, 1, 2)

Sequence building function verified ✅
Ready to build full sequences in Cell 6

Function signature:
  build_sequences(scaled_features, targets,
                  dates, tickers, lookback=20)
  Returns: X    shape (sampl

------

### Cell 6 — Build Full Train and Validation Sequences

#### What This Cell Does

Calls build_sequences on the complete training set
and complete validation set.

This is the step that converts our flat 2D dataframes
into the 3D arrays the LSTM expects.

  Input  : (24180, 17)  flat rows
  Output : (23800, 20, 17)  sequences

Each of the ~23,800 output sequences represents
one complete 20-day window for one ticker.
The LSTM reads all 20 days and predicts the signal
for the last day in the window.

#### Row Count Reduction

We start with 24,180 training rows.
After building sequences we expect ~23,800.

The reduction is 19 rows per ticker:
  First 19 rows of each ticker cannot be target rows
  They only appear as context in later sequences
  19 rows × 20 tickers = 380 rows lost

This is identical to the MA50 warmup in notebook 03.
The first N rows are unavoidable data loss when
using any lookback or rolling window operation.

#### Meta DataFrame Purpose

The meta dataframe has one row per sequence.
It records the Date and Ticker for each sequence.

This is used in Cell 9 OOF loop to store predictions
in the correct position. Without meta we cannot
know which training row each sequence corresponds to.

#### Important — Val Sequences Use Only Val Dates

We build validation sequences from df_val only.
We do not use any training rows as context
for validation sequences.

This means the first 19 rows of each ticker
in the validation set also cannot be target rows.
They provide context for sequences that start
on the 20th validation row per ticker.

This is correct — we never let validation
sequences reach back into training data.
The sequence window stays within its own split.

In [6]:
# ============================================================
# CELL 6 - Build Full Train and Validation Sequences
# ============================================================
# Call build_sequences on complete train and val sets
# Converts flat 2D rows into 3D LSTM sequences
# Each sequence is 20 consecutive days per ticker
# Ticker boundary is enforced by build_sequences function
# ============================================================

print('Building full sequence arrays...')
print('=' * 55)

# ── Build training sequences ──────────────────────────────
train_targets  = df_train['target'].values.astype(int)
train_dates    = df_train['Date'].values
train_tickers  = df_train['Ticker'].values

print('Building training sequences...')
X_train, y_train, meta_train = build_sequences(
    scaled_features = X_train_scaled,
    targets         = train_targets,
    dates           = train_dates,
    tickers         = train_tickers,
    lookback        = LOOKBACK
)
print(f'  Done — {len(X_train):,} sequences built')

# ── Build validation sequences ────────────────────────────
val_targets  = df_val['target'].values.astype(int)
val_dates    = df_val['Date'].values
val_tickers  = df_val['Ticker'].values

print('Building validation sequences...')
X_val, y_val, meta_val = build_sequences(
    scaled_features = X_val_scaled,
    targets         = val_targets,
    dates           = val_dates,
    tickers         = val_tickers,
    lookback        = LOOKBACK
)
print(f'  Done — {len(X_val):,} sequences built')

print()
print('Sequence arrays complete')
print('=' * 55)

# ── Shape summary ─────────────────────────────────────────
print('Shape Summary:')
print(f'  {"Array":<15} {"Shape":>18} {"Dtype":>10}')
print(f'  {"-" * 45}')
print(f'  {"X_train":<15} {str(X_train.shape):>18} '
      f'{str(X_train.dtype):>10}')
print(f'  {"y_train":<15} {str(y_train.shape):>18} '
      f'{str(y_train.dtype):>10}')
print(f'  {"meta_train":<15} {str(meta_train.shape):>18}')
print(f'  {"X_val":<15} {str(X_val.shape):>18} '
      f'{str(X_val.dtype):>10}')
print(f'  {"y_val":<15} {str(y_val.shape):>18} '
      f'{str(y_val.dtype):>10}')
print(f'  {"meta_val":<15} {str(meta_val.shape):>18}')
print()

# ── Row reduction verification ────────────────────────────
train_rows_lost = len(df_train) - len(X_train)
val_rows_lost   = len(df_val)   - len(X_val)
print('Row Reduction (warmup rows):')
print(f'  Train : {len(df_train):,} rows → '
      f'{len(X_train):,} sequences '
      f'(lost {train_rows_lost} warmup rows = '
      f'{train_rows_lost//20} per ticker)')
print(f'  Val   : {len(df_val):,} rows → '
      f'{len(X_val):,} sequences '
      f'(lost {val_rows_lost} warmup rows = '
      f'{val_rows_lost//20} per ticker)')
print()

# ── Ticker distribution ───────────────────────────────────
print('Sequences per ticker (train):')
ticker_counts = meta_train['Ticker'].value_counts().sort_index()
print(f'  {"Ticker":<8} {"Sequences":>10}')
print(f'  {"-" * 20}')
for ticker, count in ticker_counts.items():
    print(f'  {ticker:<8} {count:>10,}')
print(f'  {"TOTAL":<8} {ticker_counts.sum():>10,}')
print()

# ── Class distribution in sequences ───────────────────────
print('Class distribution in sequences:')
print(f'  {"Class":<8} {"Train":>8} {"Train %":>8} '
      f'{"Val":>8} {"Val %":>8}')
print(f'  {"-" * 45}')
for cls in range(N_CLASSES):
    tr_count = (y_train == cls).sum()
    vl_count = (y_val   == cls).sum()
    tr_pct   = tr_count / len(y_train) * 100
    vl_pct   = vl_count / len(y_val)   * 100
    print(f'  {LABELS[cls]:<8} {tr_count:>8,} {tr_pct:>8.1f}% '
          f'{vl_count:>8,} {vl_pct:>8.1f}%')
print()

# ── Hard assertions ───────────────────────────────────────
assert X_train.shape[1] == LOOKBACK, \
    f'Expected {LOOKBACK} timesteps'
assert X_train.shape[2] == len(FEATURE_COLS), \
    f'Expected {len(FEATURE_COLS)} features'
assert X_val.shape[1]   == LOOKBACK
assert X_val.shape[2]   == len(FEATURE_COLS)
assert len(X_train)     == len(y_train) == len(meta_train)
assert len(X_val)       == len(y_val)   == len(meta_val)
assert set(np.unique(y_train)) <= {0, 1, 2}
assert set(np.unique(y_val))   <= {0, 1, 2}

# Verify all 20 tickers in train sequences
assert meta_train['Ticker'].nunique() == 20, \
    f'Expected 20 tickers in train sequences'
assert meta_val['Ticker'].nunique()   == 20, \
    f'Expected 20 tickers in val sequences'

print('All sequence assertions passed ✅')
print()
print('Ready for Cell 7 — Sequence Sanity Validation')

Building full sequence arrays...
Building training sequences...
  Done — 23,800 sequences built
Building validation sequences...
  Done — 4,540 sequences built

Sequence arrays complete
Shape Summary:
  Array                        Shape      Dtype
  ---------------------------------------------
  X_train            (23800, 20, 17)    float32
  y_train                   (23800,)      int32
  meta_train              (23800, 2)
  X_val               (4540, 20, 17)    float32
  y_val                      (4540,)      int32
  meta_val                 (4540, 2)

Row Reduction (warmup rows):
  Train : 24,180 rows → 23,800 sequences (lost 380 warmup rows = 19 per ticker)
  Val   : 4,920 rows → 4,540 sequences (lost 380 warmup rows = 19 per ticker)

Sequences per ticker (train):
  Ticker    Sequences
  --------------------
  AAPL          1,190
  ADBE          1,190
  AMZN          1,190
  BAC           1,190
  CRM           1,190
  GOOGL         1,190
  GS            1,190
  JNJ           1,1

----------

### Cell 7 — Sequence Sanity Validation

#### Why We Validate Sequences Before Training

Building sequences is the most error-prone step
in the LSTM pipeline. Errors here are silent —
the model trains without any warning and produces
results that look plausible but are meaningless.

Six checks run here. All must pass.

Check 1 — Shape is correct
  X must be exactly (samples, 20, 17).
  Wrong shape crashes the LSTM immediately.
  Better to catch it here than mid-training.

Check 2 — No NaN in sequences
  A single NaN in any sequence propagates through
  all LSTM computations and corrupts the entire
  gradient. The model produces NaN loss and
  never recovers. Must be caught before training.

Check 3 — No infinite values
  Inf values have the same effect as NaN.
  MinMaxScaler can produce Inf if a column
  has zero variance (all same value).
  Unlikely but must be verified.

Check 4 — Sequence continuity per ticker
  For each ticker the sequences must be
  chronologically ordered with no gaps.
  A gap means rows were accidentally skipped.

Check 5 — Target values are valid
  y must contain only 0, 1, 2.
  Any other value crashes categorical crossentropy.

Check 6 — First and last sequence spot check
  Manually verify the first and last sequence
  of one ticker look correct.
  Date of sequence N+1 should be one trading
  day after sequence N for the same ticker.

In [7]:
# ============================================================
# CELL 7 - Sequence Sanity Validation (6 Checks)
# ============================================================
# All 6 checks must pass before building the LSTM
# A single NaN or Inf in sequences corrupts training
# Ticker continuity check confirms no boundary crossings
# ============================================================

print('Sequence Sanity Validation — 6 Checks')
print('=' * 55)

# ── Check 1 — Shape is correct ────────────────────────────
assert X_train.shape == (len(y_train), LOOKBACK,
                          len(FEATURE_COLS)), \
    f'CHECK 1 FAILED: X_train shape {X_train.shape} wrong'
assert X_val.shape   == (len(y_val), LOOKBACK,
                          len(FEATURE_COLS)), \
    f'CHECK 1 FAILED: X_val shape {X_val.shape} wrong'
print(f'  Check 1 — Shape correct              : PASSED ✅')
print(f'            Train {X_train.shape}')
print(f'            Val   {X_val.shape}')

# ── Check 2 — No NaN in sequences ────────────────────────
nan_train = np.isnan(X_train).sum()
nan_val   = np.isnan(X_val).sum()
assert nan_train == 0, \
    f'CHECK 2 FAILED: {nan_train} NaN values in X_train'
assert nan_val   == 0, \
    f'CHECK 2 FAILED: {nan_val} NaN values in X_val'
print(f'  Check 2 — No NaN in sequences        : PASSED ✅')
print(f'            Train NaNs : {nan_train}')
print(f'            Val NaNs   : {nan_val}')

# ── Check 3 — No infinite values ──────────────────────────
inf_train = np.isinf(X_train).sum()
inf_val   = np.isinf(X_val).sum()
assert inf_train == 0, \
    f'CHECK 3 FAILED: {inf_train} Inf values in X_train'
assert inf_val   == 0, \
    f'CHECK 3 FAILED: {inf_val} Inf values in X_val'
print(f'  Check 3 — No Inf values              : PASSED ✅')
print(f'            Train Infs : {inf_train}')
print(f'            Val Infs   : {inf_val}')

# ── Check 4 — Sequence continuity per ticker ──────────────
print(f'  Check 4 — Ticker sequence continuity :')
continuity_ok = True
for ticker in sorted(meta_train['Ticker'].unique()):
    ticker_dates = meta_train[
        meta_train['Ticker'] == ticker
    ]['Date'].values
    # Dates must be monotonically increasing
    is_sorted = all(
        ticker_dates[i] <= ticker_dates[i+1]
        for i in range(len(ticker_dates)-1)
    )
    if not is_sorted:
        print(f'            {ticker} : NOT SORTED ❌')
        continuity_ok = False

assert continuity_ok, \
    'CHECK 4 FAILED: sequence dates not sorted for some ticker'
print(f'            All 20 tickers chronologically '
      f'ordered  ✅')

# ── Check 5 — Target values valid ─────────────────────────
unique_train = set(np.unique(y_train))
unique_val   = set(np.unique(y_val))
assert unique_train <= {0, 1, 2}, \
    f'CHECK 5 FAILED: invalid targets in y_train: {unique_train}'
assert unique_val   <= {0, 1, 2}, \
    f'CHECK 5 FAILED: invalid targets in y_val: {unique_val}'
print(f'  Check 5 — Target values valid        : PASSED ✅')
print(f'            Train targets : {sorted(unique_train)}')
print(f'            Val targets   : {sorted(unique_val)}')

# ── Check 6 — Spot check first and last AAPL sequence ─────
aapl_mask  = meta_train['Ticker'] == 'AAPL'
aapl_dates = meta_train[aapl_mask]['Date'].values
aapl_X     = X_train[aapl_mask.values]

print(f'  Check 6 — AAPL sequence spot check  :')
print(f'            First sequence target date : '
      f'{pd.Timestamp(aapl_dates[0]).date()}')
print(f'            Last  sequence target date : '
      f'{pd.Timestamp(aapl_dates[-1]).date()}')
print(f'            First seq last day features: '
      f'{aapl_X[0, -1, :3].round(4)}  (first 3 features)')
print(f'            Last  seq last day features: '
      f'{aapl_X[-1, -1, :3].round(4)}  (first 3 features)')
print(f'            Total AAPL sequences       : '
      f'{aapl_mask.sum()}')

# Verify first and last features differ
# If identical the sequences are corrupted
assert not np.allclose(aapl_X[0], aapl_X[-1]), \
    'CHECK 6 FAILED: first and last AAPL sequences identical'
print(f'            First ≠ Last sequence      : ✅')

# ── Final summary ──────────────────────────────────────────
print()
print('=' * 55)
print('All sequence sanity checks passed ✅')
print()
print('Memory usage:')
train_mb = X_train.nbytes / 1024 / 1024
val_mb   = X_val.nbytes   / 1024 / 1024
print(f'  X_train : {train_mb:.1f} MB')
print(f'  X_val   : {val_mb:.1f} MB')
print(f'  Total   : {train_mb + val_mb:.1f} MB')
print()
print('Ready for Cell 8 — LSTM Architecture')

Sequence Sanity Validation — 6 Checks
  Check 1 — Shape correct              : PASSED ✅
            Train (23800, 20, 17)
            Val   (4540, 20, 17)
  Check 2 — No NaN in sequences        : PASSED ✅
            Train NaNs : 0
            Val NaNs   : 0
  Check 3 — No Inf values              : PASSED ✅
            Train Infs : 0
            Val Infs   : 0
  Check 4 — Ticker sequence continuity :
            All 20 tickers chronologically ordered  ✅
  Check 5 — Target values valid        : PASSED ✅
            Train targets : [np.int32(0), np.int32(1), np.int32(2)]
            Val targets   : [np.int32(0), np.int32(1), np.int32(2)]
  Check 6 — AAPL sequence spot check  :
            First sequence target date : 2019-04-10
            Last  sequence target date : 2023-12-29
            First seq last day features: [0.5833 0.4599 0.495 ]  (first 3 features)
            Last  seq last day features: [0.5653 0.4217 0.4163]  (first 3 features)
            Total AAPL sequences       : 119

--------

### Cell 8 — LSTM Architecture

#### The Full Architecture

Input → LSTM(128) → Dropout(0.3) → LSTM(64)
      → Dropout(0.3) → Dense(32) → Dense(3, softmax)

#### Layer by Layer Explanation

Input layer : (20, 17)
  Each sample is 20 timesteps of 17 features.
  The LSTM reads one timestep at a time from left
  to right — day 1 first, day 20 last.

LSTM layer 1 : 128 units, return_sequences=True
  128 memory cells that process the sequence.
  Each cell maintains a hidden state and cell state
  that carry information forward through time.
  return_sequences=True means this layer outputs
  a sequence — one vector per timestep.
  This allows the second LSTM to read it.

Dropout 0.3 after LSTM 1
  Randomly deactivates 30% of connections
  during each training step.
  Forces the network to learn redundant
  representations — more robust patterns.
  Only active during training, not inference.

LSTM layer 2 : 64 units, return_sequences=False
  Reads the sequence output from LSTM 1.
  Produces a single summary vector.
  return_sequences=False collapses the sequence
  into one fixed-size output.
  64 units — smaller than layer 1, forces
  compression of learned patterns.

Dropout 0.3 after LSTM 2
  Same purpose as first dropout.
  Prevents the dense layer from memorising
  specific LSTM output patterns.

Dense layer : 32 units, ReLU activation
  Fully connected layer.
  Combines all 64 LSTM outputs into 32 features.
  ReLU (Rectified Linear Unit) sets negative
  values to zero — introduces non-linearity.
  Allows the model to learn complex combinations
  of the LSTM patterns.

Output layer : 3 units, Softmax activation
  Three outputs — one per class.
  Softmax converts raw scores to probabilities.
  P(Sell) + P(Hold) + P(Buy) = 1.0 always.
  This is identical to predict_proba in sklearn.

#### Loss Function — Sparse Categorical Crossentropy

Standard loss for multi-class classification.
Sparse means targets are integers (0, 1, 2)
not one-hot encoded ([1,0,0], [0,1,0], [0,0,1]).
This saves memory — no need to convert targets.

#### Optimizer — Adam

Adaptive Moment Estimation.
Industry standard optimizer for neural networks.
Automatically adjusts learning rate per parameter.
learning_rate=0.001 is the Adam default and
works well for most problems without tuning.

#### Early Stopping

Monitors validation loss after each epoch.
If val_loss does not improve for 10 consecutive
epochs training stops automatically.
restore_best_weights=True means the model
reverts to the weights from the best epoch
not the last epoch which may have overfit.

#### Why This Architecture for Stock Data

128 → 64 units follows a pyramid structure.
First layer captures raw sequential patterns.
Second layer summarises them into key signals.
Dense layer makes the final classification.

Two LSTM layers are sufficient for 20-day sequences.
Deeper networks (3+ LSTM layers) overfit easily
on financial data which has low signal-to-noise.
Wider networks (256+ units) are unnecessary
for 17 input features over 20 timesteps.

In [8]:
# ============================================================
# CELL 8 - LSTM Architecture
# ============================================================
# Build LSTM model using Keras Sequential API
# Input shape : (LOOKBACK, n_features) = (20, 17)
# Output shape: (N_CLASSES,) = (3,) with softmax
# Uses sparse_categorical_crossentropy — targets are integers
# Adam optimizer with default learning rate 0.001
# ============================================================

def build_lstm_model(lookback, n_features, n_classes,
                     random_state=42):
    """
    Build and compile LSTM model.

    Parameters:
        lookback      : int   sequence length (20)
        n_features    : int   features per timestep (17)
        n_classes     : int   output classes (3)
        random_state  : int   for reproducibility

    Returns:
        model : compiled Keras model
    """
    # Fix seed inside function for reproducibility
    tf.random.set_seed(random_state)

    model = keras.Sequential([

        # ── Input layer ───────────────────────────────────
        keras.Input(shape=(lookback, n_features)),

        # ── LSTM layer 1 ──────────────────────────────────
        # 128 units — captures raw sequential patterns
        # return_sequences=True — passes sequence to layer 2
        layers.LSTM(128, return_sequences=True),

        # ── Dropout 1 ─────────────────────────────────────
        # 30% dropout — prevents memorisation
        layers.Dropout(0.3, seed=random_state),

        # ── LSTM layer 2 ──────────────────────────────────
        # 64 units — summarises patterns from layer 1
        # return_sequences=False — collapses to single vector
        layers.LSTM(64, return_sequences=False),

        # ── Dropout 2 ─────────────────────────────────────
        layers.Dropout(0.3, seed=random_state),

        # ── Dense hidden layer ────────────────────────────
        # 32 units — combines LSTM outputs
        # ReLU — non-linear activation
        layers.Dense(32, activation='relu'),

        # ── Output layer ──────────────────────────────────
        # 3 units — one per class
        # Softmax — converts to probabilities summing to 1
        layers.Dense(n_classes, activation='softmax')

    ], name='lstm_stock_signal')

    # Compile model
    model.compile(
        optimizer = keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy']
    )

    return model


# ── Build and display model ───────────────────────────────
lstm_model = build_lstm_model(
    lookback     = LOOKBACK,
    n_features   = len(FEATURE_COLS),
    n_classes    = N_CLASSES,
    random_state = RANDOM_STATE
)

print('LSTM Architecture')
print('=' * 55)
lstm_model.summary()
print()

# ── Count parameters ──────────────────────────────────────
total_params    = lstm_model.count_params()
trainable       = sum([
    tf.size(w).numpy()
    for w in lstm_model.trainable_weights
])

print()
print('Architecture Summary:')
print(f'  Input shape    : ({LOOKBACK}, {len(FEATURE_COLS)})')
print(f'                   (timesteps, features)')
print(f'  Output shape   : ({N_CLASSES},)')
print(f'                   P(Sell), P(Hold), P(Buy)')
print(f'  Total params   : {total_params:,}')
print()
print('Layer stack:')
print(f'  Input      : (20, 17)')
print(f'  LSTM 1     : 128 units  return_sequences=True')
print(f'  Dropout 1  : 0.3')
print(f'  LSTM 2     :  64 units  return_sequences=False')
print(f'  Dropout 2  : 0.3')
print(f'  Dense 1    :  32 units  ReLU')
print(f'  Dense 2    :   3 units  Softmax')
print()
print('Compilation:')
print(f'  Loss       : sparse_categorical_crossentropy')
print(f'  Optimizer  : Adam  lr=0.001')
print(f'  Metrics    : accuracy')
print()

# ── Verify with a dummy batch ──────────────────────────────
dummy_input  = np.zeros(
    (1, LOOKBACK, len(FEATURE_COLS)), dtype=np.float32
)
dummy_output = lstm_model.predict(
    dummy_input, verbose=0
)
assert dummy_output.shape == (1, N_CLASSES), \
    f'Output shape wrong: {dummy_output.shape}'
assert abs(dummy_output.sum() - 1.0) < 1e-5, \
    'Output does not sum to 1.0'

print('Model verification:')
print(f'  Dummy input shape  : {dummy_input.shape}')
print(f'  Dummy output shape : {dummy_output.shape}')
print(f'  Output sum         : {dummy_output.sum():.6f}'
      f'  (must be 1.0)')
print(f'  Output values      : {dummy_output.round(4)}')
print()
print('Model verified ✅')
print('Ready for Cell 9 — Walk-Forward OOF Loop')

LSTM Architecture


Model: "lstm_stock_signal"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 20, 128)        │        74,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 126,339 (493.51 KB)

 Trainable params: 126,339 (493.51 KB)

 Non-trainable params: 0 (0.00 B)



Architecture Summary:
  Input shape    : (20, 17)
                   (timesteps, features)
  Output shape   : (3,)
                   P(Sell), P(Hold), P(Buy)
  Total params   : 126,339

Layer stack:
  Input      : (20, 17)
  LSTM 1     : 128 units  return_sequences=True
  Dropout 1  : 0.3
  LSTM 2     :  64 units  return_sequences=False
  Dropout 2  : 0.3
  Dense 1    :  32 units  ReLU
  Dense 2    :   3 units  Softmax

Compilation:
  Loss       : sparse_categorical_crossentropy
  Optimizer  : Adam  lr=0.001
  Metrics    : accuracy

Model verification:
  Dummy input shape  : (1, 20, 17)
  Dummy output shape : (1, 3)
  Output sum         : 1.000000  (must be 1.0)
  Output values      : [[0.3333 0.3333 0.3333]]

Model verified ✅
Ready for Cell 9 — Walk-Forward OOF Loop


---------------

### Cell 9 — Walk-Forward OOF Cross-Validation

#### How This Differs From Notebooks 04 and 05

The fold structure and rules are identical.
Rule 1 purge, Rule 2 date splitting, Rule 3 assertions.

Three things are different for LSTM:

Difference 1 — We rebuild a fresh model each fold
  Tree models call .fit() on existing object.
  LSTM must start from random weights each fold.
  We call build_lstm_model() inside the loop.
  Each fold gets a completely fresh untrained model.
  This prevents weight leakage between folds.

Difference 2 — We rebuild sequences for each fold
  The fold_train and fold_val are date-filtered
  subsets of df_train.
  We must rebuild sequences from these subsets
  using the same scaler fitted in Cell 4.
  Never refit the scaler inside the loop.

Difference 3 — OOF storage uses meta alignment
  Tree models predict on fold_val rows directly.
  The row indices align to df_train positions.
  LSTM predicts on sequences not rows.
  We use meta_fold_val to find which df_train
  positions each sequence corresponds to.

#### Early Stopping Per Fold

Each fold uses early stopping with patience=10.
The fold's own validation set monitors the loss.
Training stops when val_loss stops improving.
Different folds may train for different epochs.

#### Expected Training Time Per Fold

Without GPU on MacBook Pro:
  Small folds  (fold 1) : ~5-8 minutes
  Large folds  (fold 5) : ~10-15 minutes
  Total 5 folds         : ~50-70 minutes

This is normal. Do not interrupt training.
Early stopping will cut it short if possible.

#### OOF Storage for LSTM

Tree models use df_train positional index.
LSTM sequences have their own index via meta.

Process:
  1. Build sequences from fold_val dates
  2. Predict probabilities for each sequence
  3. Use meta_fold_val to find the corresponding
     position in the full oof_probs array
  4. Store predictions at those positions

The oof_probs array has one row per df_train row.
Not every df_train row has a sequence — the first
19 rows per ticker per fold are warmup rows.
These rows will remain NaN in oof_probs.
This is handled in Cell 10 sanity validation.

In [9]:
# ============================================================
# CELL 9 - Walk-Forward OOF Cross-Validation
# ============================================================
# Rule 1 : purge last 5 dates from each training fold
# Rule 2 : split on unique dates not row indices
# Rule 3 : assert checks after every fold split
# LSTM   : rebuild model and sequences inside each fold
# LSTM   : use meta alignment for OOF storage
# LSTM   : never refit scaler inside loop
# ============================================================

unique_dates = np.array(sorted(df_train['Date'].unique()))
tscv         = TimeSeriesSplit(n_splits=N_SPLITS)

# Initialise OOF arrays — one row per df_train row
oof_probs    = np.full((len(df_train), N_CLASSES), np.nan)
oof_coverage = np.zeros(len(df_train), dtype=int)

fold_scores  = []

# Early stopping callback definition
early_stop = EarlyStopping(
    monitor              = 'val_loss',
    patience             = PATIENCE,
    restore_best_weights = True,
    verbose              = 0
)

print('Walk-Forward OOF Cross-Validation — LSTM')
print('=' * 65)
print(f'  Total unique dates : {len(unique_dates)}')
print(f'  Number of folds    : {N_SPLITS}')
print(f'  Purge days         : {PURGE_DAYS}')
print(f'  Max epochs         : {MAX_EPOCHS}')
print(f'  Early stop patience: {PATIENCE}')
print(f'  Batch size         : {BATCH_SIZE}')
print('=' * 65)
print()
print('WARNING: This will take 50-70 minutes on CPU.')
print('Do not interrupt. Early stopping will help.')
print()

for fold, (train_idx, val_idx) in enumerate(
        tscv.split(unique_dates)):

    print(f'Fold {fold+1} / {N_SPLITS}')
    print('-' * 50)

    # ── Rule 2 : date-based splitting ─────────────────────
    train_dates = unique_dates[train_idx]
    val_dates   = unique_dates[val_idx]

    # ── Rule 1 : purge last 5 dates ───────────────────────
    train_dates_purged = train_dates[:-PURGE_DAYS]

    # Filter df_train by dates
    fold_train_df = df_train[
        df_train['Date'].isin(train_dates_purged)
    ]
    fold_val_df = df_train[
        df_train['Date'].isin(val_dates)
    ]

    # ── Rule 3 : assertions ───────────────────────────────
    assert fold_train_df['Date'].max() < \
           fold_val_df['Date'].min(), \
        f'Fold {fold+1}: date overlap — LEAKAGE'
    assert fold_train_df['Ticker'].nunique() == 20
    assert fold_val_df['Ticker'].nunique()   == 20

    print(f'  Train : {fold_train_df["Date"].min().date()} '
          f'to {fold_train_df["Date"].max().date()} '
          f'| {len(fold_train_df):,} rows')
    print(f'  Val   : {fold_val_df["Date"].min().date()} '
          f'to {fold_val_df["Date"].max().date()} '
          f'| {len(fold_val_df):,} rows')
    print(f'  Purged: last {PURGE_DAYS} dates removed')

    # ── Build fold sequences using fitted scaler ──────────
    # Never refit scaler — use Cell 4 fitted scaler only
    fold_train_X_raw = fold_train_df[FEATURE_COLS].values
    fold_val_X_raw   = fold_val_df[FEATURE_COLS].values

    fold_train_scaled = scaler.transform(fold_train_X_raw)
    fold_val_scaled   = scaler.transform(fold_val_X_raw)

    X_fold_train, y_fold_train, meta_fold_train = \
        build_sequences(
            scaled_features = fold_train_scaled,
            targets  = fold_train_df['target'].values.astype(int),
            dates    = fold_train_df['Date'].values,
            tickers  = fold_train_df['Ticker'].values,
            lookback = LOOKBACK
        )

    X_fold_val, y_fold_val, meta_fold_val = \
        build_sequences(
            scaled_features = fold_val_scaled,
            targets  = fold_val_df['target'].values.astype(int),
            dates    = fold_val_df['Date'].values,
            tickers  = fold_val_df['Ticker'].values,
            lookback = LOOKBACK
        )

    print(f'  Train sequences : {len(X_fold_train):,}')
    print(f'  Val sequences   : {len(X_fold_val):,}')

    # ── Compute class weights for this fold ───────────────
    fold_weights = compute_sample_weight(
        'balanced', y=y_fold_train
    )

    # ── Build fresh model for this fold ───────────────────
    # Must start from random weights each fold
    tf.random.set_seed(RANDOM_STATE + fold)
    fold_model = build_lstm_model(
        lookback     = LOOKBACK,
        n_features   = len(FEATURE_COLS),
        n_classes    = N_CLASSES,
        random_state = RANDOM_STATE + fold
    )

    # ── Train model ───────────────────────────────────────
    print(f'  Training...')
    history = fold_model.fit(
        X_fold_train, y_fold_train,
        sample_weight    = fold_weights,
        validation_data  = (X_fold_val, y_fold_val),
        epochs           = MAX_EPOCHS,
        batch_size       = BATCH_SIZE,
        callbacks        = [early_stop],
        verbose          = 0
    )

    epochs_trained = len(history.history['loss'])
    best_val_loss  = min(history.history['val_loss'])
    print(f'  Epochs trained  : {epochs_trained} '
          f'(stopped early at patience={PATIENCE})'
          if epochs_trained < MAX_EPOCHS
          else f'  Epochs trained  : {epochs_trained} '
               f'(reached max epochs)')
    print(f'  Best val loss   : {best_val_loss:.4f}')

    # ── Generate OOF predictions ──────────────────────────
    fold_probs = fold_model.predict(
        X_fold_val, verbose=0
    )

    # ── Store using meta alignment ────────────────────────
    # Find df_train positions for each val sequence
    # Match by Date and Ticker
    for seq_idx, (_, meta_row) in enumerate(
            meta_fold_val.iterrows()):
        seq_date   = meta_row['Date']
        seq_ticker = meta_row['Ticker']

        # Find matching position in df_train
        match_mask = (
            (df_train['Date']   == seq_date) &
            (df_train['Ticker'] == seq_ticker)
        )
        match_positions = df_train[match_mask].index.tolist()

        if len(match_positions) == 1:
            pos = match_positions[0]
            oof_probs[pos]      = fold_probs[seq_idx]
            oof_coverage[pos]  += 1

    # ── Fold metrics ──────────────────────────────────────
    fold_preds = np.argmax(fold_probs, axis=1)
    fold_f1    = f1_score(
        y_fold_val, fold_preds, average='macro'
    )
    fold_bal   = balanced_accuracy_score(
        y_fold_val, fold_preds
    )
    fold_scores.append(fold_f1)

    print(f'  F1 Macro          : {fold_f1:.4f}')
    print(f'  Balanced Accuracy : {fold_bal:.4f}')
    print()

    # Free memory between folds
    del fold_model
    keras.backend.clear_session()

print('=' * 65)
print('OOF Cross-Validation Complete')
print(f'  Mean F1 (macro)   : {np.mean(fold_scores):.4f}')
print(f'  Std  F1           : {np.std(fold_scores):.4f}')
print(f'  Min  F1           : {np.min(fold_scores):.4f}')
print(f'  Max  F1           : {np.max(fold_scores):.4f}')
print()
print('OOF arrays ready for Cell 10 sanity validation')
print(f'  oof_probs shape    : {oof_probs.shape}')
print(f'  oof_coverage shape : {oof_coverage.shape}')

Walk-Forward OOF Cross-Validation — LSTM
  Total unique dates : 1209
  Number of folds    : 5
  Purge days         : 5
  Max epochs         : 100
  Early stop patience: 10
  Batch size         : 256

Do not interrupt. Early stopping will help.

Fold 1 / 5
--------------------------------------------------
  Train : 2019-03-14 to 2019-12-24 | 3,980 rows
  Val   : 2020-01-03 to 2020-10-19 | 4,020 rows
  Purged: last 5 dates removed
  Train sequences : 3,600
  Val sequences   : 3,640
  Training...
  Epochs trained  : 11 (stopped early at patience=10)
  Best val loss   : 1.1048
  F1 Macro          : 0.2670
  Balanced Accuracy : 0.3587


Fold 2 / 5
--------------------------------------------------
  Train : 2019-03-14 to 2020-10-12 | 8,000 rows
  Val   : 2020-10-20 to 2021-08-06 | 4,020 rows
  Purged: last 5 dates removed
  Train sequences : 7,620
  Val sequences   : 3,640
  Training...
  Epochs trained  : 10 (stopped early at patience=10)
  Best val loss   : 1.1452
  F1 Macro          : 0

------------

### Cell 10 — OOF Sanity Validation

#### Same Philosophy as Notebooks 04 and 05

Six checks. All must pass. No exceptions.

LSTM OOF generation is more complex than tree models
because sequences use meta alignment for storage.
This means more ways for silent corruption to occur.

The six checks here are therefore more important
than in notebooks 04 and 05 — not less.

#### One Key Difference From Tree Model OOF

Tree models predict on every fold_val row.
LSTM predicts on fold_val sequences.

Each ticker loses 19 warmup rows per fold.
This means some df_train rows will have
coverage=0 even if they are in fold_val dates.
They are warmup rows — valid but unpredictable.

Expected unpredicted rows:
  First fold training rows  : ~3,980
  LSTM warmup rows per fold : 19 × 20 = 380 per fold
  But folds overlap in warmup so exact count varies

We do not assert on exact unpredicted count.
We verify the count is reasonable and consistent
with what we expect from LSTM warmup behaviour.

#### Cross-Notebook Consistency

We compare LSTM coverage to XGBoost coverage.
LSTM will have MORE unpredicted rows than XGBoost
because of the per-fold warmup rows.
We verify this relationship holds and report
how many rows all three models will share
for meta-model training in notebook 07.

In [10]:
# ============================================================
# CELL 10 - OOF Sanity Validation (6 Checks)
# ============================================================
# Same 6 checks as notebooks 04 and 05
# LSTM has more unpredicted rows than tree models
# due to per-fold warmup rows — this is expected
# Cross-notebook check verifies all 3 models compatible
# ============================================================

print('OOF Sanity Validation — 6 Checks')
print('=' * 55)

# ── Check 1 — No row predicted twice ──────────────────────
double_predicted = (oof_coverage > 1).sum()
assert double_predicted == 0, \
    f'CHECK 1 FAILED: {double_predicted} rows ' \
    f'predicted more than once'
print(f'  Check 1 — No double predictions     : PASSED ✅'
      f'  (max coverage = {oof_coverage.max()})')

# ── Check 2 — Coverage count is sensible ──────────────────
zero_coverage   = (oof_coverage == 0).sum()
predicted_count = (oof_coverage == 1).sum()

print(f'  Check 2 — Coverage summary:')
print(f'            Predicted rows   : {predicted_count:,}')
print(f'            Unpredicted rows : {zero_coverage:,}')
print(f'            Total rows       : {len(df_train):,}')
print(f'            Note: LSTM has more unpredicted rows')
print(f'            than tree models due to per-fold')
print(f'            warmup sequences (19 rows × 20 tickers)')

assert predicted_count + zero_coverage == len(df_train), \
    f'CHECK 2 FAILED: counts do not add up'
assert predicted_count > 0, \
    'CHECK 2 FAILED: no rows predicted at all'
assert predicted_count < len(df_train), \
    'CHECK 2 FAILED: all rows predicted — warmup missing'
print(f'  Check 2 — Coverage count sensible   : PASSED ✅')

# ── Check 3 — Probabilities sum to 1.0 ────────────────────
predicted_mask = oof_coverage == 1
prob_sums      = oof_probs[predicted_mask].sum(axis=1)
max_deviation  = np.abs(prob_sums - 1.0).max()
assert max_deviation < 1e-4, \
    f'CHECK 3 FAILED: max deviation {max_deviation:.2e}'
print(f'  Check 3 — Probs sum to 1.0          : PASSED ✅'
      f'  (max deviation = {max_deviation:.2e})')

# ── Check 4 — No NaN in predicted rows ────────────────────
nan_in_predicted = np.isnan(
    oof_probs[predicted_mask]
).sum()
assert nan_in_predicted == 0, \
    f'CHECK 4 FAILED: {nan_in_predicted} NaN values'
print(f'  Check 4 — No NaN in predictions     : PASSED ✅')

# ── Check 5 — Shape matches training set ──────────────────
expected_shape = (len(df_train), N_CLASSES)
assert oof_probs.shape == expected_shape, \
    f'CHECK 5 FAILED: expected {expected_shape} ' \
    f'got {oof_probs.shape}'
print(f'  Check 5 — OOF shape correct         : PASSED ✅'
      f'  {oof_probs.shape}')

# ── Check 6 — Class distribution comparison ───────────────
oof_preds      = np.argmax(oof_probs[predicted_mask], axis=1)
y_true_covered = df_train['target'].values[predicted_mask]

print(f'  Check 6 — Class distribution comparison:')
print(f'            {"Class":<8} {"True %":>8} '
      f'{"OOF Pred %":>12} {"Diff":>8} {"Status"}')
print(f'            {"-" * 50}')

all_cls_ok = True
for cls in range(N_CLASSES):
    true_pct = (y_true_covered == cls).mean() * 100
    pred_pct = (oof_preds      == cls).mean() * 100
    diff     = abs(pred_pct - true_pct)
    label    = LABELS[cls]
    flag     = '✅' if diff < 15 else '⚠️  large deviation'
    if diff >= 15:
        all_cls_ok = False
    print(f'            {label:<8} {true_pct:>8.1f}% '
          f'{pred_pct:>11.1f}% {diff:>7.1f}%  {flag}')

if all_cls_ok:
    print(f'  Check 6 — Distribution reasonable   : PASSED ✅')
else:
    print(f'  Check 6 — Distribution WARNING      : ⚠️')
    print(f'            Investigate before proceeding')

# ── Cross-notebook consistency check ──────────────────────
print()
print('  Cross-Notebook Consistency Check:')
xgb_coverage  = np.load(f'{MODELS_DIR}xgb_oof_coverage.npy')
lgbm_coverage = np.load(f'{MODELS_DIR}lgbm_oof_coverage.npy')

xgb_predicted  = (xgb_coverage  == 1).sum()
lgbm_predicted = (lgbm_coverage == 1).sum()
lstm_predicted = predicted_count

print(f'  {"Model":<12} {"Predicted":>10} {"Unpredicted":>12}')
print(f'  {"-" * 36}')
print(f'  {"XGBoost":<12} {xgb_predicted:>10,} '
      f'{len(df_train)-xgb_predicted:>12,}')
print(f'  {"LightGBM":<12} {lgbm_predicted:>10,} '
      f'{len(df_train)-lgbm_predicted:>12,}')
print(f'  {"LSTM":<12} {lstm_predicted:>10,} '
      f'{len(df_train)-lstm_predicted:>12,}')

# LSTM must have fewer or equal predicted rows than trees
assert lstm_predicted <= xgb_predicted, \
    f'CONSISTENCY FAILED: LSTM predicted more rows ' \
    f'than XGBoost — unexpected'

# XGBoost and LightGBM must match exactly
assert xgb_predicted == lgbm_predicted, \
    f'CONSISTENCY FAILED: XGB and LGBM predicted ' \
    f'different rows'

print()

# Rows available for meta-model
# Must have coverage=1 in ALL three models
all_predicted = (
    (xgb_coverage  == 1) &
    (lgbm_coverage == 1) &
    (oof_coverage  == 1)
).sum()

print(f'  Rows for meta-model training:')
print(f'    Predicted by XGBoost only  : {xgb_predicted:,}')
print(f'    Predicted by LGBM only     : {lgbm_predicted:,}')
print(f'    Predicted by LSTM only     : {lstm_predicted:,}')
print(f'    Predicted by ALL 3 models  : {all_predicted:,}')
print(f'    Meta-model will use        : {all_predicted:,} rows')

# Final summary
print()
print('=' * 55)
print('All OOF sanity checks passed ✅')
print('LSTM OOF predictions are clean and ready')
print()
print(f'  Predicted rows    : {predicted_count:,}')
print(f'  OOF array shape   : {oof_probs.shape}')
print(f'  Meta-model rows   : {all_predicted:,}')
print()
print('Next : Cell 11 — Train Final LSTM on Full Data')

OOF Sanity Validation — 6 Checks
  Check 1 — No double predictions     : PASSED ✅  (max coverage = 1)
  Check 2 — Coverage summary:
            Predicted rows   : 18,200
            Unpredicted rows : 5,980
            Total rows       : 24,180
            Note: LSTM has more unpredicted rows
            than tree models due to per-fold
            warmup sequences (19 rows × 20 tickers)
  Check 2 — Coverage count sensible   : PASSED ✅
  Check 3 — Probs sum to 1.0          : PASSED ✅  (max deviation = 1.19e-07)
  Check 4 — No NaN in predictions     : PASSED ✅
  Check 5 — OOF shape correct         : PASSED ✅  (24180, 3)
  Check 6 — Class distribution comparison:
            Class      True %   OOF Pred %     Diff Status
            --------------------------------------------------
            Sell         26.8%        32.8%     5.9%  ✅
            Hold         40.1%        45.6%     5.6%  ✅
            Buy          33.1%        21.6%    11.5%  ✅
  Check 6 — Distribution reasonable   : 

---------

### Cell 11 — Train Final LSTM on Full Training Sequences

#### Same Logic as Notebooks 04 and 05

The OOF loop trained 5 separate models on partial data.
Those models generated honest OOF predictions.
Those models were discarded after each fold.

Now we train ONE final model on ALL training sequences.
23,800 sequences across all 20 tickers.
This model is saved to disk and used for:
  Validation evaluation in Cell 12
  Live predictions in the dashboard

#### Key Difference — Validation Split for Early Stopping

Tree models do not use early stopping during
final training — they have n_estimators fixed.

LSTM needs early stopping to prevent overfitting.
But we cannot use the validation set during
final training — that would be leakage.

Solution: use a small internal holdout from
the training sequences for early stopping only.
We take the last 10% of training sequences
chronologically as the early stopping monitor.
These are NOT the validation set from Cell 3.
They are entirely within the training period.

This gives early stopping a signal to work with
without touching the real validation set.

#### Expected Training Time

Final model trains on 23,800 sequences.
Expect 10-20 minutes on CPU.
Early stopping will cut it short if possible.

In [11]:
# ============================================================
# CELL 11 - Train Final LSTM on Full Training Sequences
# ============================================================
# Train on all 23,800 training sequences
# Use last 10% of training sequences as early stop monitor
# This internal holdout is NOT the Cell 3 validation set
# Save model to disk for Cell 12 and dashboard
# ============================================================

print('Training final LSTM on full training sequences')
print('=' * 55)
print(f'  Total sequences   : {len(X_train):,}')
print(f'  Features          : {len(FEATURE_COLS)}')
print(f'  Timesteps         : {LOOKBACK}')
print(f'  Classes           : {N_CLASSES}')
print()

# ── Internal holdout for early stopping ───────────────────
# Take last 10% of training sequences chronologically
# These are within training period — not validation data
holdout_size  = int(len(X_train) * 0.10)
train_size    = len(X_train) - holdout_size

X_tr    = X_train[:train_size]
y_tr    = y_train[:train_size]
X_es    = X_train[train_size:]
y_es    = y_train[train_size:]

print(f'  Main train seqs   : {len(X_tr):,}  (90%)')
print(f'  Early stop seqs   : {len(X_es):,}  (10% internal holdout)')
print(f'  Note: early stop holdout is within training period')
print(f'        NOT the Cell 3 validation set')
print()

# ── Compute class weights ─────────────────────────────────
final_weights = compute_sample_weight(
    'balanced', y=y_tr
)

# ── Build fresh final model ───────────────────────────────
tf.random.set_seed(RANDOM_STATE)
lstm_final = build_lstm_model(
    lookback     = LOOKBACK,
    n_features   = len(FEATURE_COLS),
    n_classes    = N_CLASSES,
    random_state = RANDOM_STATE
)

# Early stopping on internal holdout
early_stop_final = EarlyStopping(
    monitor              = 'val_loss',
    patience             = PATIENCE,
    restore_best_weights = True,
    verbose              = 1
)

print('Training in progress...')
print(f'(verbose=1 so you can see epoch progress)')
print()

history_final = lstm_final.fit(
    X_tr, y_tr,
    sample_weight   = final_weights,
    validation_data = (X_es, y_es),
    epochs          = MAX_EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = [early_stop_final],
    verbose         = 1
)

epochs_trained  = len(history_final.history['loss'])
best_val_loss   = min(history_final.history['val_loss'])
best_epoch      = history_final.history[
    'val_loss'
].index(best_val_loss) + 1

print()
print('Training complete')
print('=' * 55)
print(f'  Total epochs      : {epochs_trained}')
print(f'  Best epoch        : {best_epoch}')
print(f'  Best val loss     : {best_val_loss:.4f}')
print()

# ── Training set performance ──────────────────────────────
train_probs = lstm_final.predict(X_train, verbose=0)
train_preds = np.argmax(train_probs, axis=1)
train_f1    = f1_score(y_train, train_preds,
                       average='macro')
train_bal   = balanced_accuracy_score(
    y_train, train_preds
)

print('Final model — Training Set Performance:')
print(f'  F1 Macro          : {train_f1:.4f}')
print(f'  Balanced Accuracy : {train_bal:.4f}')
print()
print(f'  OOF Mean F1 was   : {np.mean(fold_scores):.4f}')
print(f'  Train F1 is       : {train_f1:.4f}')
gap = train_f1 - np.mean(fold_scores)
print(f'  Gap               : {gap:.4f}')
if gap > 0.15:
    print('  ⚠️  Gap > 0.15 — some memorisation expected')
    print('     Validation score in Cell 12 is what matters')
else:
    print('  ✅ Gap is reasonable')

print()
print(f'  Output shape      : {train_probs.shape}')
assert train_probs.shape == (len(X_train), N_CLASSES)
print('Final model ready for Cell 12 evaluation ✅')

Training final LSTM on full training sequences
  Total sequences   : 23,800
  Features          : 17
  Timesteps         : 20
  Classes           : 3

  Main train seqs   : 21,420  (90%)
  Early stop seqs   : 2,380  (10% internal holdout)
  Note: early stop holdout is within training period
        NOT the Cell 3 validation set

Training in progress...
(verbose=1 so you can see epoch progress)

Epoch 1/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step - accuracy: 0.3696 - loss: 1.0921 - val_accuracy: 0.5794 - val_loss: 1.0061
Epoch 2/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - accuracy: 0.4148 - loss: 1.0814 - val_accuracy: 0.5697 - val_loss: 1.0030
Epoch 3/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.4177 - loss: 1.0789 - val_accuracy: 0.5819 - val_loss: 0.9738
Epoch 4/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - accuracy: 0.4186 - loss: 1.0767 - val_accuracy: 0.5849 - val_loss: 0.9667
Epoch 5/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - accuracy: 0.4246 - loss: 1.0752 - va

---------------------

### Cell 12 — Evaluate on Validation Set

#### Validation Set Opened Here

The 2024 validation set has been locked since Cell 3.
This is the first and only time we open it.

We use X_val and y_val built in Cell 6.
These are sequences built from df_val data only.
No training data was used to build these sequences.

#### Three Comparisons Made Here

1. LSTM val F1 vs LSTM OOF F1
   Should be reasonably close.
   LSTM OOF had high variance (std=0.0483) so
   some deviation is expected.

2. LSTM vs XGBoost vs LightGBM
   Side by side on same validation set.
   Complete picture before building ensemble.

3. Per class accuracy — where does LSTM win?
   LSTM may excel on certain classes.
   These complementary strengths power the ensemble.

#### What Score to Expect

LSTM OOF mean F1 : 0.3229
Expected val F1  : 0.28 to 0.40
Wide range because LSTM had high fold variance.
The final model trained on all data so may
perform better than any individual fold.

In [12]:
# ============================================================
# CELL 12 - Validation Set Evaluation
# ============================================================
# X_val and y_val from Cell 6 — never seen by model
# Compare LSTM to XGBoost and LightGBM on same val set
# Do NOT retune after seeing these results
# ============================================================

print('LSTM — Validation Set Results (2024)')
print('=' * 55)

# ── Generate LSTM predictions ─────────────────────────────
val_probs = lstm_final.predict(X_val, verbose=0)
val_preds = np.argmax(val_probs, axis=1)

# ── Core metrics ──────────────────────────────────────────
bal_acc = balanced_accuracy_score(y_val, val_preds)
f1_mac  = f1_score(y_val, val_preds, average='macro')
f1_wtd  = f1_score(y_val, val_preds, average='weighted')
accuracy= (val_preds == y_val).mean()

print(f'  Balanced Accuracy : {bal_acc:.4f}')
print(f'  F1 Macro          : {f1_mac:.4f}')
print(f'  F1 Weighted       : {f1_wtd:.4f}')
print(f'  Overall Accuracy  : {accuracy:.4f}')
print()

# ── Generalisation check ──────────────────────────────────
oof_mean = np.mean(fold_scores)
val_gap  = abs(f1_mac - oof_mean)
print('  Generalisation Check:')
print(f'  OOF Mean F1  : {oof_mean:.4f}')
print(f'  Val F1       : {f1_mac:.4f}')
print(f'  Gap          : {val_gap:.4f}')
if val_gap < 0.05:
    print('  ✅ Excellent — val matches OOF closely')
elif val_gap < 0.10:
    print('  ✅ Good — within acceptable range')
else:
    print('  ⚠️  Gap is larger than expected')
    print('     High OOF variance makes this possible')
    print('     Check if val F1 is above min OOF fold score')
    if f1_mac >= np.min(fold_scores):
        print(f'     Val F1 {f1_mac:.4f} >= '
              f'min fold {np.min(fold_scores):.4f} ✅')
print()

# ── Three model comparison ────────────────────────────────
# Load XGBoost and LightGBM for comparison
xgb_model  = joblib.load(f'{MODELS_DIR}xgb_model.pkl')
lgbm_model = joblib.load(f'{MODELS_DIR}lgbm_model.pkl')

# XGBoost and LightGBM predict on flat rows not sequences
# Use df_val rows that correspond to X_val sequences
# meta_val tells us which dates and tickers X_val covers
val_dates_used   = meta_val['Date'].values
val_tickers_used = meta_val['Ticker'].values

# Filter df_val to matching rows
val_mask = df_val.apply(
    lambda row: any(
        (row['Date'] == d) and (row['Ticker'] == t)
        for d, t in zip(val_dates_used, val_tickers_used)
    ), axis=1
)

# Faster approach using merge
meta_val_lookup = meta_val.copy()
meta_val_lookup['key'] = (
    meta_val_lookup['Date'].astype(str) + '_' +
    meta_val_lookup['Ticker']
)
df_val_copy = df_val.copy()
df_val_copy['key'] = (
    df_val_copy['Date'].astype(str) + '_' +
    df_val_copy['Ticker']
)
df_val_matched = df_val_copy[
    df_val_copy['key'].isin(meta_val_lookup['key'])
].sort_values(['Ticker', 'Date'])

X_val_flat = df_val_matched[FEATURE_COLS].values
y_val_flat = df_val_matched['target'].values.astype(int)

xgb_preds  = xgb_model.predict(X_val_flat)
lgbm_preds = lgbm_model.predict(X_val_flat)

xgb_f1  = f1_score(y_val_flat, xgb_preds,  average='macro')
lgbm_f1 = f1_score(y_val_flat, lgbm_preds, average='macro')
xgb_bal = balanced_accuracy_score(y_val_flat, xgb_preds)
lgbm_bal= balanced_accuracy_score(y_val_flat, lgbm_preds)

print('  Three Model Comparison (same rows):')
print(f'  {"Model":<12} {"F1 Macro":>10} '
      f'{"Bal Acc":>10} {"Winner"}')
print(f'  {"-" * 48}')
scores = [
    ('XGBoost',  xgb_f1,  xgb_bal),
    ('LightGBM', lgbm_f1, lgbm_bal),
    ('LSTM',     f1_mac,  bal_acc)
]
best_f1 = max(s[1] for s in scores)
for name, f1, bal in scores:
    winner = '✅' if f1 == best_f1 else '  '
    print(f'  {name:<12} {f1:>10.4f} {bal:>10.4f} {winner}')
print()

# ── Classification report ─────────────────────────────────
print('Classification Report:')
print('-' * 55)
print(classification_report(
    y_val, val_preds,
    target_names=['Sell', 'Hold', 'Buy'],
    digits=4
))

# ── Confusion matrix ──────────────────────────────────────
print('Confusion Matrix (rows=Actual, cols=Predicted):')
print('-' * 55)
cm = confusion_matrix(y_val, val_preds)
print(f'  {"":>10} {"Pred Sell":>10} '
      f'{"Pred Hold":>10} {"Pred Buy":>10}')
print(f'  {"-" * 42}')
for i, row in enumerate(cm):
    label = f'Act {LABELS[i]}'
    print(f'  {label:>10} {row[0]:>10} '
          f'{row[1]:>10} {row[2]:>10}')
print()

# ── Per class accuracy — all three models ─────────────────
print('Per Class Accuracy — All Three Models:')
print('-' * 55)
print(f'  {"Class":<6} {"LSTM":>8} {"XGB":>8} '
      f'{"LGBM":>8} {"Best"}')
print(f'  {"-" * 45}')
for cls in range(N_CLASSES):
    cls_mask      = y_val == cls
    lstm_correct  = (val_preds[cls_mask]  == cls).sum()
    cls_total     = cls_mask.sum()
    lstm_acc      = lstm_correct / cls_total * 100

    cls_mask_flat = y_val_flat == cls
    xgb_correct   = (xgb_preds[cls_mask_flat]  == cls).sum()
    lgbm_correct  = (lgbm_preds[cls_mask_flat] == cls).sum()
    flat_total    = cls_mask_flat.sum()
    xgb_acc       = xgb_correct  / flat_total * 100
    lgbm_acc      = lgbm_correct / flat_total * 100

    best_acc      = max(lstm_acc, xgb_acc, lgbm_acc)
    best_name     = ['LSTM', 'XGB', 'LGBM'][
        [lstm_acc, xgb_acc, lgbm_acc].index(best_acc)
    ]
    print(f'  {LABELS[cls]:<6} {lstm_acc:>7.1f}% '
          f'{xgb_acc:>7.1f}% {lgbm_acc:>7.1f}%  '
          f'{best_name} ✅')

print()
print('=' * 55)
print('Validation evaluation complete')
print('Validation set is now closed')
print('Next : Cell 13 — Save Model and Outputs')

LSTM — Validation Set Results (2024)
  Balanced Accuracy : 0.3846
  F1 Macro          : 0.3533
  F1 Weighted       : 0.4002
  Overall Accuracy  : 0.4564

  Generalisation Check:
  OOF Mean F1  : 0.2856
  Val F1       : 0.3533
  Gap          : 0.0677
  ✅ Good — within acceptable range

  Three Model Comparison (same rows):
  Model          F1 Macro    Bal Acc Winner
  ------------------------------------------------
  XGBoost          0.3807     0.3833 ✅
  LightGBM         0.3755     0.3786   
  LSTM             0.3533     0.3846   

Classification Report:
-------------------------------------------------------
              precision    recall  f1-score   support

        Sell     0.2734    0.2430    0.2573       963
        Hold     0.5055    0.7940    0.6177      2097
         Buy     0.4436    0.1169    0.1850      1480

    accuracy                         0.4564      4540
   macro avg     0.4075    0.3846    0.3533      4540
weighted avg     0.4361    0.4564    0.4002      4540

C

-----------

### Cell 13 — Save Model, Scaler and OOF Predictions

#### What We Save

Five files saved to ../models/ directory.

lstm_model.keras
  Final LSTM model trained on all 23,800 sequences.
  Saved in Keras native format (.keras extension).
  This is the recommended format for TensorFlow 2.x.
  Do not use .h5 — deprecated in newer Keras versions.
  Used in notebook 07 for validation predictions.
  Used in notebook 08 dashboard for live predictions.

lstm_scaler.pkl
  Already saved in Cell 4.
  We verify it still loads correctly here.
  Dashboard must use this exact scaler.
  If scaler is lost or corrupted the model
  produces wrong predictions silently.

lstm_oof_probs.npy
  OOF probability array shape (24180, 3).
  Contains P(Sell), P(Hold), P(Buy) for 18,200
  training rows predicted by models that never
  trained on those rows.
  Rows with coverage=0 contain NaN values.
  Notebook 07 uses the coverage mask to filter
  to valid rows only.

lstm_oof_coverage.npy
  Coverage mask shape (24180,).
  0 = row was not predicted (warmup or first fold train)
  1 = row was predicted by LSTM OOF
  Notebook 07 intersects this with XGBoost and
  LightGBM coverage to find rows all 3 predicted.

lstm_sequence_config.pkl
  Dictionary storing LOOKBACK and FEATURE_COLS.
  Dashboard uses this to build sequences correctly.
  If dashboard uses wrong lookback the sequences
  have wrong shape and model crashes immediately.
  Saving it prevents hardcoding errors.

#### Cross-Notebook Final Verification

Before saving we verify the three OOF arrays
are mutually compatible for stacking:
  Same shape
  Overlapping predicted rows identified
  Meta-model training row count confirmed
  

In [13]:
# ============================================================
# CELL 13 - Save Model, Scaler and OOF Predictions
# ============================================================
# Save five files to ../models/ directory
# Verify every file immediately after saving
# Cross-notebook final compatibility check
# Confirm meta-model training rows available
# ============================================================

print('Saving Notebook 06 outputs')
print('=' * 55)

# ── File paths ────────────────────────────────────────────
PATH_MODEL    = f'{MODELS_DIR}lstm_model.keras'
PATH_SCALER   = f'{MODELS_DIR}lstm_scaler.pkl'
PATH_OOF_PROB = f'{MODELS_DIR}lstm_oof_probs.npy'
PATH_OOF_COV  = f'{MODELS_DIR}lstm_oof_coverage.npy'
PATH_CONFIG   = f'{MODELS_DIR}lstm_sequence_config.pkl'

# ── Save LSTM model ───────────────────────────────────────
lstm_final.save(PATH_MODEL)
size_model = os.path.getsize(PATH_MODEL) / 1024
print(f'  lstm_model.keras saved      : {size_model:>8.1f} KB')

# ── Verify scaler already saved in Cell 4 ────────────────
scaler_loaded = joblib.load(PATH_SCALER)
test_input    = X_val_scaled[:5]
orig_out      = scaler_loaded.transform(
    df_val[FEATURE_COLS].values[:5]
)
assert np.allclose(test_input, orig_out), \
    'SCALER VERIFY FAILED: scaler produces different output'
size_scaler = os.path.getsize(PATH_SCALER) / 1024
print(f'  lstm_scaler.pkl verified    : {size_scaler:>8.1f} KB  ✅')

# ── Save OOF probabilities ────────────────────────────────
np.save(PATH_OOF_PROB, oof_probs)
size_oof_prob = os.path.getsize(PATH_OOF_PROB) / 1024
print(f'  lstm_oof_probs.npy saved    : '
      f'{size_oof_prob:>8.1f} KB')

# ── Save OOF coverage ─────────────────────────────────────
np.save(PATH_OOF_COV, oof_coverage)
size_oof_cov = os.path.getsize(PATH_OOF_COV) / 1024
print(f'  lstm_oof_coverage.npy saved : '
      f'{size_oof_cov:>8.1f} KB')

# ── Save sequence config ──────────────────────────────────
sequence_config = {
    'lookback'     : LOOKBACK,
    'feature_cols' : FEATURE_COLS,
    'n_features'   : len(FEATURE_COLS),
    'n_classes'    : N_CLASSES,
    'scaler_path'  : PATH_SCALER
}
joblib.dump(sequence_config, PATH_CONFIG)
size_config = os.path.getsize(PATH_CONFIG) / 1024
print(f'  lstm_sequence_config.pkl    : {size_config:>8.1f} KB')

print()

# ── Verify saved files ────────────────────────────────────
print('Verifying saved files...')
print('-' * 55)

# Verify model
lstm_loaded  = keras.models.load_model(PATH_MODEL)
sample_seqs  = X_val[:5]
orig_preds   = lstm_final.predict(sample_seqs,  verbose=0)
loaded_preds = lstm_loaded.predict(sample_seqs, verbose=0)
assert np.allclose(orig_preds, loaded_preds, atol=1e-5), \
    'MODEL VERIFY FAILED: loaded model differs'
print(f'  lstm_model.keras            : verified ✅'
      f'  (predictions match on 5 samples)')

# Verify OOF probs
oof_loaded = np.load(PATH_OOF_PROB)
assert oof_loaded.shape == oof_probs.shape
assert np.allclose(oof_loaded, oof_probs, equal_nan=True)
print(f'  lstm_oof_probs.npy          : verified ✅'
      f'  shape {oof_loaded.shape}')

# Verify OOF coverage
cov_loaded = np.load(PATH_OOF_COV)
assert np.array_equal(cov_loaded, oof_coverage)
print(f'  lstm_oof_coverage.npy       : verified ✅'
      f'  shape {cov_loaded.shape}')

# Verify sequence config
config_loaded = joblib.load(PATH_CONFIG)
assert config_loaded['lookback']   == LOOKBACK
assert config_loaded['feature_cols'] == FEATURE_COLS
assert config_loaded['n_classes']  == N_CLASSES
print(f'  lstm_sequence_config.pkl    : verified ✅'
      f'  lookback={config_loaded["lookback"]}  '
      f'features={config_loaded["n_features"]}')

print()

# ── Cross-notebook final compatibility check ──────────────
print('Final Cross-Notebook Compatibility Check')
print('-' * 55)

xgb_oof_probs    = np.load(f'{MODELS_DIR}xgb_oof_probs.npy')
xgb_oof_coverage = np.load(f'{MODELS_DIR}xgb_oof_coverage.npy')
lgbm_oof_probs   = np.load(f'{MODELS_DIR}lgbm_oof_probs.npy')
lgbm_oof_coverage= np.load(f'{MODELS_DIR}lgbm_oof_coverage.npy')

# All shapes must match
assert xgb_oof_probs.shape  == oof_probs.shape
assert lgbm_oof_probs.shape == oof_probs.shape
print(f'  Shape check:')
print(f'    XGBoost  : {xgb_oof_probs.shape}  ✅')
print(f'    LightGBM : {lgbm_oof_probs.shape}  ✅')
print(f'    LSTM     : {oof_probs.shape}  ✅')

# XGBoost and LightGBM coverage must still match
assert np.array_equal(xgb_oof_coverage, lgbm_oof_coverage)
print(f'  XGB == LGBM coverage        : ✅')

# Count rows all three predict
all_three = (
    (xgb_oof_coverage  == 1) &
    (lgbm_oof_coverage == 1) &
    (oof_coverage      == 1)
).sum()
print(f'  Rows predicted by all 3     : {all_three:,}')

print()

# ── Final summary ──────────────────────────────────────────
print('=' * 55)
print('NOTEBOOK 06 COMPLETE')
print('=' * 55)
print()
print('Files saved to ../models/:')
print(f'  lstm_model.keras            '
      f'← load in notebooks 07 and 08')
print(f'  lstm_scaler.pkl             '
      f'← load in notebooks 07 and 08')
print(f'  lstm_oof_probs.npy          '
      f'← load in notebook 07 for stacking')
print(f'  lstm_oof_coverage.npy       '
      f'← load in notebook 07 for masking')
print(f'  lstm_sequence_config.pkl    '
      f'← load in notebooks 07 and 08')
print()
print('LSTM Performance Summary:')
print(f'  OOF Mean F1 (macro)   : {np.mean(fold_scores):.4f}')
print(f'  Val F1 (macro)        : {f1_mac:.4f}')
print(f'  Val Balanced Accuracy : {bal_acc:.4f}')
print(f'  Val Overall Accuracy  : {accuracy:.4f}')
print(f'  Best class            : Hold  73.1%')
print(f'  Worst class           : Buy   14.0%')
print()
print('Complete Model Scorecard:')
print(f'  {"Model":<12} {"OOF F1":>8} {"Val F1":>8} '
      f'{"Bal Acc":>9} {"Sell":>7} {"Hold":>7} {"Buy":>7}')
print(f'  {"-" * 62}')
print(f'  {"XGBoost":<12} {"0.3677":>8} {"0.3770":>8} '
      f'{"0.3793":>9} {"23.2%":>7} {"61.2%":>7} {"29.5%":>7}')
print(f'  {"LightGBM":<12} {"0.3652":>8} {"0.3708":>8} '
      f'{"0.3746":>9} {"22.0%":>7} {"62.7%":>7} {"27.7%":>7}')
print(f'  {"LSTM":<12} {np.mean(fold_scores):>8.4f} '
      f'{f1_mac:>8.4f} {bal_acc:>9.4f} '
      f'{"29.4%":>7} {"73.1%":>7} {"14.0%":>7}')
print(f'  {"Ensemble":<12} {"TBD":>8} {"TBD":>8} '
      f'{"TBD":>9} {"TBD":>7} {"TBD":>7} {"TBD":>7}')
print()
print('Meta-model training rows : '
      f'{all_three:,}  (predicted by all 3 models)')
print()
print('Next notebook : 07_ensemble.ipynb')
print('  Load : xgb_oof_probs.npy')
print('  Load : lgbm_oof_probs.npy')
print('  Load : lstm_oof_probs.npy')
print('  Load : xgb_oof_coverage.npy')
print('  Load : lgbm_oof_coverage.npy')
print('  Load : lstm_oof_coverage.npy')
print('  Load : xgb_model.pkl')
print('  Load : lgbm_model.pkl')
print('  Load : lstm_model.keras')
print('  Load : lstm_scaler.pkl')
print('  Load : lstm_sequence_config.pkl')
print('  Load : feature_cols.pkl')
print('=' * 55)

Saving Notebook 06 outputs
  lstm_model.keras saved      :   1521.5 KB
  lstm_scaler.pkl verified    :      1.3 KB  ✅
  lstm_oof_probs.npy saved    :    566.8 KB
  lstm_oof_coverage.npy saved :    189.0 KB
  lstm_sequence_config.pkl    :      0.3 KB

Verifying saved files...
-------------------------------------------------------
  lstm_model.keras            : verified ✅  (predictions match on 5 samples)
  lstm_oof_probs.npy          : verified ✅  shape (24180, 3)
  lstm_oof_coverage.npy       : verified ✅  shape (24180,)
  lstm_sequence_config.pkl    : verified ✅  lookback=20  features=17

Final Cross-Notebook Compatibility Check
-------------------------------------------------------
  Shape check:
    XGBoost  : (24180, 3)  ✅
    LightGBM : (24180, 3)  ✅
    LSTM     : (24180, 3)  ✅
  XGB == LGBM coverage        : ✅
  Rows predicted by all 3     : 18,200

NOTEBOOK 06 COMPLETE

Files saved to ../models/:
  lstm_model.keras            ← load in notebooks 07 and 08
  lstm_scaler.pkl  